In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",   # important for output_attentions=True
).eval()

In [ ]:
import torch
import torch.nn.functional as F
from typing import Optional

In [ ]:
def get_llava_image_span_from_inputs(model, processor, input_ids, pixel_values):
    """
    Estimate image token span after HF LLaVA expands <image> into visual patch tokens.

    For llava-hf/llava-1.5-7b-hf:
      input_ids contain one <image> token.
      The internal merged sequence replaces it with N image patch features.
    """

    image_token_id = getattr(model.config, "image_token_index", None)

    if image_token_id is None:
        image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

    image_positions = (input_ids[0] == image_token_id).nonzero(as_tuple=False)

    if len(image_positions) == 0:
        raise ValueError("Could not find <image> token in input_ids.")

    image_token_pos = int(image_positions[0].item())

    # Compute number of image features from vision tower.
    with torch.inference_mode():
        vision_outputs = model.vision_tower(
            pixel_values,
            output_hidden_states=True,
            return_dict=True,
        )

        selected = vision_outputs.hidden_states[model.config.vision_feature_layer]

        # LLaVA default usually drops CLS when strategy == "default"
        if getattr(model.config, "vision_feature_select_strategy", "default") == "default":
            selected = selected[:, 1:]

        num_image_tokens = selected.shape[1]

    image_start = image_token_pos
    image_end = image_start + num_image_tokens
    response_start = input_ids.shape[1] - 1 + num_image_tokens

    return {
        "image_start": image_start,
        "image_end": image_end,
        "response_start": response_start,
    }

In [ ]:
def compute_opera_overtrust_penalty(
    attentions,
    key_position,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    """
    Compute OPERA-style over-trust penalty from decoder self-attention.

    attentions[-1]: last decoder layer attention.
    Shape is usually [B, num_heads, query_len, key_len].
    We inspect the last generated/probed token's attention to previous response tokens.
    """

    if attentions is None:
        raise ValueError(
            "Model did not return attentions. Reload with attn_implementation='eager'."
        )

    last_layer_attn = attentions[-1]  # [B, H, Q, K]
    attn = last_layer_attn[:, :, -1, :].mean(dim=1)  # [B, K]

    response_start = key_position["response_start"]
    seq_len = attn.shape[-1]

    if response_start >= seq_len - 1:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    # Generated text tokens before current token.
    response_attn = attn[:, response_start:seq_len - 1]

    if response_attn.numel() == 0:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    max_summary_attn = response_attn.max(dim=-1).values

    # OPERA thresholding style: scale attention, penalize only if above threshold.
    overtrust = torch.relu(max_summary_attn * scale_factor - threshold)

    penalty = penalty_weights * overtrust / scale_factor

    return penalty

In [ ]:
@torch.inference_mode()
def opera_generate_one_hf(
    model,
    processor,
    image,
    question: str = "Describe this image.",
    max_new_tokens: int = 256,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    model.eval()

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    prompt = f"USER: <image>\n{question}\nASSISTANT:"

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding=True,
    )

    processor.tokenizer.padding_side = old_padding_side

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype)

    key_position = get_llava_image_span_from_inputs(
        model=model,
        processor=processor,
        input_ids=input_ids,
        pixel_values=pixel_values,
    )

    generated_ids = input_ids.clone()
    prompt_len = input_ids.shape[1]

    eos_token_id = model.generation_config.eos_token_id

    if eos_token_id is None:
        eos_token_ids = []
    elif isinstance(eos_token_id, int):
        eos_token_ids = [eos_token_id]
    else:
        eos_token_ids = list(eos_token_id)

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = processor.tokenizer.eos_token_id

    # Initial prompt forward
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    if outputs.attentions is None:
        raise ValueError(
            "outputs.attentions is None. Reload model with attn_implementation='eager'."
        )

    past_key_values = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]

    for step in range(max_new_tokens):
        log_probs = F.log_softmax(next_logits, dim=-1)

        k = min(num_attn_candidates, log_probs.shape[-1])
        top_scores, top_tokens = torch.topk(log_probs, k=k, dim=-1)

        candidate_final_scores = []
        candidate_tokens = []
        candidate_outputs = []

        for cand_idx in range(k):
            cand_token = top_tokens[:, cand_idx:cand_idx + 1]  # [1, 1]
            cand_log_score = top_scores[:, cand_idx]           # [1]

            cand_attention_mask = torch.cat(
                [
                    attention_mask,
                    torch.ones(
                        (attention_mask.shape[0], 1),
                        dtype=attention_mask.dtype,
                        device=attention_mask.device,
                    ),
                ],
                dim=-1,
            )

            cand_outputs = model(
                input_ids=cand_token,
                attention_mask=cand_attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
                output_attentions=True,
                return_dict=True,
            )

            penalty = compute_opera_overtrust_penalty(
                attentions=cand_outputs.attentions,
                key_position=key_position,
                scale_factor=scale_factor,
                threshold=threshold,
                penalty_weights=penalty_weights,
            )

            adjusted_score = cand_log_score - penalty

            candidate_final_scores.append(adjusted_score)
            candidate_tokens.append(cand_token)
            candidate_outputs.append(cand_outputs)

        candidate_final_scores = torch.stack(candidate_final_scores, dim=1)  # [1, K]
        best_idx = int(torch.argmax(candidate_final_scores, dim=-1).item())

        next_token = candidate_tokens[best_idx]
        selected_outputs = candidate_outputs[best_idx]

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        past_key_values = selected_outputs.past_key_values
        next_logits = selected_outputs.logits[:, -1, :]

        if len(eos_token_ids) > 0 and int(next_token.item()) in eos_token_ids:
            break

    new_tokens = generated_ids[:, prompt_len:]

    answer = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    return answer

In [ ]:
def opera_generate_batch_hf(
    model,
    processor,
    images,
    question: str = "Describe this image.",
    max_new_tokens: int = 256,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    single_input = not isinstance(images, list)

    if single_input:
        images = [images]

    outputs = []

    for image in images:
        answer = opera_generate_one_hf(
            model=model,
            processor=processor,
            image=image,
            question=question,
            max_new_tokens=max_new_tokens,
            num_attn_candidates=num_attn_candidates,
            scale_factor=scale_factor,
            threshold=threshold,
            penalty_weights=penalty_weights,
        )

        outputs.append(answer)

    if single_input:
        return outputs[0]

    return outputs

In [ ]:
import os
import json
import gc
from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [ ]:
def load_coco_val2017_images(
    coco_img_dir: str = "data/coco2017/val2017",
    coco_ann_dir: str = "data/coco2017/annotations",
    ann_file: str = "instances_val2017.json",
):
    ann_path = Path(coco_ann_dir) / ann_file

    if not ann_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {ann_path}")

    with open(ann_path, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    items = []

    for img_info in coco_data["images"]:
        img_path = Path(coco_img_dir) / img_info["file_name"]

        if not img_path.exists():
            print(f"Missing image: {img_path}")
            continue

        items.append({
            "id": int(img_info["id"]),
            "file_name": img_info["file_name"],
            "img_path": str(img_path),
        })

    items = sorted(items, key=lambda x: x["id"])

    print(f"Loaded {len(items)} COCO val2017 images")

    return items


def save_json_atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    tmp_path.replace(path)


def clean_output(text):
    text = text.strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()

In [ ]:
def infer_coco_val2017_opera(
    model,
    processor,
    coco_img_dir: str = "data/coco2017/val2017",
    coco_ann_dir: str = "data/coco2017/annotations",
    output_json_path: str = "outputs/llava15_opera_coco_val2017.json",
    ann_file: str = "instances_val2017.json",
    question: str = "Describe this image.",
    batch_size: int = 1,
    max_new_tokens: int = 256,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
    resume: bool = True,
    save_every: int = 1,
):
    """
    Output JSON format:
    [
      {
        "id": 139,
        "response": "..."
      }
    ]
    """

    items = load_coco_val2017_images(
        coco_img_dir=coco_img_dir,
        coco_ann_dir=coco_ann_dir,
        ann_file=ann_file,
    )

    results = []
    done_ids = set()

    output_json_path = Path(output_json_path)

    if resume and output_json_path.exists():
        with open(output_json_path, "r", encoding="utf-8") as f:
            results = json.load(f)

        done_ids = set(int(row["id"]) for row in results)

        print(f"Resume mode: loaded {len(results)} existing responses.")

    remaining_items = [
        item for item in items
        if int(item["id"]) not in done_ids
    ]

    print(f"Remaining images: {len(remaining_items)}")

    num_batches = (len(remaining_items) + batch_size - 1) // batch_size

    for batch_idx in tqdm(range(num_batches), desc="COCO val2017 OPERA inference"):
        start = batch_idx * batch_size
        end = start + batch_size
        batch_items = remaining_items[start:end]

        images = []
        valid_items = []

        for item in batch_items:
            try:
                image = Image.open(item["img_path"]).convert("RGB")
                images.append(image)
                valid_items.append(item)
            except Exception as e:
                print(f"Failed to load {item['img_path']}: {e}")

        if len(images) == 0:
            continue

        responses = opera_generate_batch_hf(
            model=model,
            processor=processor,
            images=images,
            question=question,
            max_new_tokens=max_new_tokens,
            num_attn_candidates=num_attn_candidates,
            scale_factor=scale_factor,
            threshold=threshold,
            penalty_weights=penalty_weights,
        )

        for item, response in zip(valid_items, responses):
            results.append({
                "id": int(item["id"]),
                "response": clean_output(response),
            })

        if (batch_idx + 1) % save_every == 0:
            results = sorted(results, key=lambda x: int(x["id"]))
            save_json_atomic(output_json_path, results)

        del images
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results = sorted(results, key=lambda x: int(x["id"]))
    save_json_atomic(output_json_path, results)

    print(f"Saved {len(results)} responses to {output_json_path}")

    return results

In [ ]:
results = infer_coco_val2017_opera(
    model=model,
    processor=processor,
    coco_img_dir="../data/coco2017/val2017",
    coco_ann_dir="../data/coco2017/annotations",
    output_json_path="../results/infer/coco_val2017/llava15/opera.json",
    ann_file="instances_val2017.json",
    question="Describe this image.",
    batch_size=16,
    max_new_tokens=256,
    num_attn_candidates=5,
    scale_factor=50.0,
    threshold=15.0,
    penalty_weights=1.0,
    resume=True,
    save_every=1,
)